In [ ]:
import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from robusta_hmf import Robusta
import time
#%matplotlib widget

In [ ]:
#define constants
c = 299792.458 # km/s
LN10 = np.log(10.)
MAX_IVAR = 2.5e3
MIN_IVAR = 0.1
H_ALPHA, H_BETA = 6564.614, 4862.721 #Reference: from classic.sdss.org
HE_I_6680 = 6680 #Reference: Johanna brain
HE_I_4026 = 4026
HE_I_4471 = 4471
HE_I_4922 = 4922
HE_I_4713 = 4713
HE_II_4686 = 4686 #Reference: Johanna brain
HE_II_4542 = 4542
HE_II_4200 = 4200


#lines
lines_balmer = [H_ALPHA, H_BETA]
lines_HE_I = [HE_I_6680, HE_I_4026, HE_I_4471, HE_I_4922, HE_I_4713]
lines_HE_II = [HE_II_4686, HE_II_4542, HE_II_4200]
all_lines = np.concatenate([lines_balmer, lines_HE_I, lines_HE_II])
mask_lines = lines_balmer

#line labes
line_labels_balmer = ["Halpha", "Hbeta"]
line_labels_HE_I = ["HeI_6680", "HeI_4026", "HeI_4471", "HeI_4922", "HeI_4713"]
line_labels_HE_II = ["HeII_4686", "HeII_4542", "HeII_4200"]
all_line_labels = np.concatenate([line_labels_balmer, line_labels_HE_I, line_labels_HE_II])

statistics = ["EW", "shift", "width"]
moment_names = ["EW", "M1", "M2", "M3", "M4"]


H_delta_lnlam_line = 1000 / c 
H_delta_loglam_line = H_delta_lnlam_line / LN10 
HE_delta_lnlam_line = 500 / c
HE_delta_loglam_line = HE_delta_lnlam_line / LN10
# log_H_ALPHA, log_H_BETA, log_HE_I, log_HE_II = np.log10(H_ALPHA), np.log10(H_BETA), np.log10(HE_I), np.log10(HE_II)

#delta loglam line for labeling
delta_balmer = np.zeros(len(lines_balmer)) + (1000 / c) / LN10 
delta_HE_I = np.zeros(len(lines_HE_I)) + (500 / c) / LN10 
delta_HE_II = np.zeros(len(lines_HE_II)) + (500 / c) / LN10 
all_delta_loglams = np.concatenate([delta_balmer, delta_HE_I, delta_HE_II])


#indexing is super important for these three arrays 
print(all_lines.shape, all_line_labels.shape, all_delta_loglams.shape)
print(all_lines, all_line_labels, all_delta_loglams)

H_delta_lnlam_line = 1000 / c 
H_delta_loglam_line = H_delta_lnlam_line / LN10 
HE_delta_lnlam_line = 500 / c
HE_delta_loglam_line = HE_delta_lnlam_line / LN10
log_H_ALPHA, log_H_BETA = np.log10(H_ALPHA), np.log10(H_BETA)


half_wids = [H_delta_loglam_line, H_delta_loglam_line, HE_delta_loglam_line, HE_delta_loglam_line] 

#THIS MUST BE ASCENDING ORDER
temp_cuts = np.array([10000, 15000, 25000, np.inf])
mod_list = [0,1]

In [ ]:
print(all_lines)

In [ ]:
#load stellar parameters
DATESTR = "2026-07-20"

stars = pd.read_parquet(f"parent_stars10k_{DATESTR}.parquet")
print("num of stars in parquet:", stars.shape[0])

spectra_temp = pd.read_parquet(f"spectra_{DATESTR}_table.parquet")
print("num of spectra in parquet:", stars.shape[0])

print(stars.shape)

In [ ]:
print("spectra before:", spectra_temp.shape)
spectra = spectra_temp.join(stars[["GAIA_ID", "Teff_fit", "NANA_HASH"]].set_index("GAIA_ID"), on = "GAIA_ID", rsuffix = "_stars", how = "left", validate = "m:1")
print("spectra after:", spectra.shape)

In [ ]:
nana = "nana_"
count = 0
for l, line in enumerate(all_line_labels):
    col_name = nana + line + "_"
    print(col_name)
    for s, stat in enumerate(moment_names):
        ##add column here
            #print(col_name + stat)
            spectra[col_name + stat] = 0.
            count+=1
            spectra[col_name + stat + "_err"] = np.inf
            count+=1
            # spectra[col_name + stat + "_resid"] = 0. ASK HOGG ABOUT THIS, JOHANNA WANTED EW MEASUREMENTS OF THE RESID
            # spectra[col_name + stat + "_resid_err"] = 0.

print(spectra.columns.to_list())
print("You added", count, "amount of columns. Does it match (num of lines) * (num of moments) * (2)?")

In [ ]:
# sanity check contents of file
print(spectra[["GAIA_ID", "NANA_HASH", "Teff_fit", "EWobs_Halpha", "rv_hydrogen_all_mean", "SPEC_FILE"]])

In [ ]:
#read in the data and deal with radial velocities

#take the ALL spectra (like 35,000)
with open(f'spectra_{DATESTR}_data.pkl','rb') as f:
    pkl_data = pickle.load(f)
print(len(pkl_data))

In [ ]:
#pkl of ALL spectra (Teff > 10000)
# BUG: This code is very brittle
fluxes, loglam, ivars, continuua, _, _, _, _, _ = pkl_data
print(fluxes.shape, loglam.shape, ivars.shape, continuua.shape)

In [ ]:
# hard stop if we SUCK
assert len(fluxes) == len(spectra)

In [ ]:
# shift to rest frame.
# Note: Only does integer-pixel shifts
# Note: Data have been slightly corrupted; delta_log_lam is not consistent across the wavelength grid.

def get_delta_log_lam(loglambdas):
    return np.median(loglambdas[1:] - loglambdas[:-1])
    
def pixel_shift(flxs, ivrs, contins, rvs, log_lambda):

    delta_log_lambda_pixel = get_delta_log_lam(log_lambda)

    # Catch bad rv values, goddammit
    bad = np.logical_not(np.isfinite(rvs))
    rad_vels = rvs.copy()
    rad_vels[bad] = 0.

    delta_log_lambdas = (rad_vels / c) / LN10
    delta_pixels = np.round(delta_log_lambdas/delta_log_lambda_pixel).astype(int)

    rest_flxs = np.zeros_like(flxs) + 1.
    rest_ivrs = np.zeros_like(ivrs)
    rest_contins = np.zeros_like(contins) + np.nan

    for i, dp in enumerate(delta_pixels):
        if dp < 0:
            rest_flxs[i, -dp:] = flxs[i, :dp]
            rest_ivrs[i, -dp:] = ivrs[i, :dp]
            rest_contins[i, -dp:] = contins[i, :dp]
            
        elif dp > 0:
            rest_flxs[i, :-dp] = flxs[i, dp:]
            rest_ivrs[i, :-dp] = ivrs[i, dp:]
            rest_contins[i, :-dp] = contins[i, dp:]
            
        else:
            rest_flxs[i, :] = flxs[i, :]
            rest_ivrs[i, :] = ivrs[i, :]
            rest_contins[i, :] = contins[i, :]

    # dammit
    rest_ivrs[bad, :] = 0.
    print("zeroing out ivars on", np.sum(bad), "stars with bad rvs")
    return rest_flxs, rest_ivrs, rest_contins

In [ ]:
# shift everything to the rest frame
rest_fluxes, rest_ivars, rest_continuua = pixel_shift(fluxes, ivars, continuua, spectra["rv_hydrogen_all_mean"].to_numpy(), loglam)
print(rest_fluxes.shape)

In [ ]:
def fix_nans_and_infinites(rest_flxs, rest_ivrs):

    #fix nans and infinities
    bad = np.logical_not(np.isfinite(rest_flxs))
    rest_flxs[bad] = 1
    rest_ivrs[bad] = 0
    
    bad = np.logical_not(np.isfinite(rest_ivrs))
    rest_flxs[bad] = 1
    rest_ivrs[bad] = 0
    
    bad = np.logical_or((rest_flxs > 2.0), (rest_flxs < 0))
    rest_flxs[bad] = 1
    rest_ivrs[bad] = 0
    
    bad = rest_ivrs > MAX_IVAR
    # rest_fluxes[bad] = 1
    # rest_ivars[bad] = 0
    rest_ivrs[bad] = MAX_IVAR
    
    bad = rest_ivrs < MIN_IVAR
    rest_flxs[bad] = 1.0
    rest_ivrs[bad] = MIN_IVAR

    return rest_flxs, rest_ivrs

In [ ]:
data, weights = fix_nans_and_infinites(rest_fluxes, rest_ivars)
print(data.shape, weights.shape, rest_continuua.shape)

In [ ]:
# make a sanity plot
tempposts = np.exp(np.linspace(np.log(np.min(spectra["Teff_fit"].to_numpy())), np.log(np.max(spectra["Teff_fit"].to_numpy())), 16))
f = plt.figure(figsize=(12, 12))
for t, T in enumerate(tempposts):
    # find closest high SNR spectrum
    highsnr = np.arange(len(spectra))[np.median(rest_ivars[:, :2000], axis=1) > 500.0]
    ii = np.argmin(np.abs(spectra["Teff_fit"].to_numpy()[highsnr] - T))
    ii = highsnr[ii]
    Teff = spectra["Teff_fit"].iloc[ii]
    print(t, T, ii, Teff)
    # plot it
    plt.step(10. ** loglam, rest_fluxes[ii] + t, c="k", where="mid")
    gid = spectra["GAIA_ID"].iloc[ii]
    label = f"{gid}; Teff = {Teff:5.0f}"
    plt.text(4000., 1.1 + t, label)
plt.ylim(0., len(tempposts) + 1.)
plt.xlim(4000, 5000)
#plt.xlim(H_ALPHA - 200, H_ALPHA + 200)

lines_balmer = [H_ALPHA, H_BETA]
lines_HE_I = [HE_I_6680, HE_I_4026, HE_I_4471, HE_I_4922, HE_I_4713]
lines_HE_II = [HE_II_4686, HE_II_4542, HE_II_4200]

for l, line in enumerate(lines_balmer):
    if l == 0:
        plt.axvline(line, label = "Balmer lines", lw=1, alpha=0.5, zorder=-10, color = "green")
    else:
        plt.axvline(line, lw=1, alpha=0.5, zorder=-10, color = "green")
        
for l, line in enumerate(lines_HE_I):
    if l == 0: 
        plt.axvline(line, label = "HeI lines", lw=1, alpha=0.5, zorder=-10, color = "orange")
    else:
        plt.axvline(line, lw=1, alpha=0.5, zorder=-10, color = "orange")
        
for l, line in enumerate(lines_HE_II):
    if l == 0: 
        plt.axvline(line, label = "HeII lines", lw=1, alpha=0.3, zorder=-10, color = "red")
    
    plt.axvline(line, lw=1, alpha=0.3, zorder=-10, color = "red")

plt.legend()    
plt.title("selected high SNR example stars, ordered by Teff")

In [ ]:
def get_good_ew_indices(spectra_df):
    #cut on Johanna's and Nanas' ews
    ews_j = spectra_df['EWobs_Halpha'].to_numpy()
    ews_n = spectra_df['nana_Halpha_EW'].to_numpy()
    good_ew_mask = (ews_j > 0) & (ews_n < 1)
    print(len(good_ew_mask))
    print(f"Johanna and Nana EW: {good_ew_mask.sum()} / {len(good_ew_mask)} spectra")

    return np.argwhere(good_ew_mask)

In [ ]:
good_ew_indxs = get_good_ew_indices(spectra)

In [ ]:
def idkyet(spectra_df, Tcuts, rest_flxes, rest_ivrs, lines, log_lambdas,  K = 12, scale = 1, nu = 1, mod_list = [0,1]):

    model_dict = {}
    synths = np.zeros_like(rest_flxes) + np.nan
    spec_temps = spectra_df['Teff_fit'].to_numpy() 

    #censoring
    log_lines = np.log10(lines)
    log_H_ALPHA, log_H_BETA = log_lines[0], log_lines[1]
    region_alpha = np.abs(log_lambdas - log_H_ALPHA)
    region_beta = np.abs(log_lambdas - log_H_BETA)
    censor_mask = np.ones_like(log_lambdas)
    censor_mask[(region_alpha < H_delta_loglam_line) | (region_beta < H_delta_loglam_line) ] = 0 #?

    for m in mod_list:
        for t in range(len(Tcuts) - 1):

            #get training and testing indices
            train_indx = ((spectra_df['NANA_HASH'].to_numpy()%2 == m) &
                          (spec_temps >= Tcuts[t]) &
                          (spectra_df['EWobs_Halpha'].to_numpy() > 0.0) &
                          (spectra_df['nana_Halpha_EW'].to_numpy() < 1.0))

            test_indx = ((spectra_df['NANA_HASH'].to_numpy()%2 != m) &
                         (spec_temps >= Tcuts[t]) &
                         (spec_temps < Tcuts[t + 1]))

            print(f"For temp {Tcuts[t]} and mod {m}: There are {train_indx.sum()} spectra in TRAIN and {test_indx.sum()} spectra in TEST")
            print(f"Model: temp {Tcuts[t]} and mod {m}")
            model = Robusta(rank=K, robust=True, robust_scale = scale, robust_nu = nu)
            start = time.perf_counter()
            model.fit(rest_flxes[train_indx], rest_ivrs[train_indx], max_iter=10000)
            end = time.perf_counter()
            print("total time:", (end - start)/60, "minutes")

            #synthesize 
            state, _ = model.infer(rest_flxes[test_indx], rest_ivrs[test_indx] * censor_mask[None, :])
            synths[test_indx] = model.synthesize(state)

            model_dict[(m, t)] = (model, train_indx, test_indx)
            print(synths[test_indx].shape)

    return synths, model_dict

In [ ]:
idkyet(spectra, temp_cuts, data, weights, all_lines, loglam)